In [ ]:
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_community.chat_models.oci_generative_ai import ChatOCIGenAI
from langchain.utilities import WikipediaAPIWrapper
from IPython.display import Markdown, display
from LoadProperties import LoadProperties

# --- Define Tools ---
tools = [
    Tool("Calculator", lambda x: str(eval(x)), "Solve math like '17 * 24'."),
    Tool("Wikipedia", WikipediaAPIWrapper().run, "Answer factual questions from Wikipedia.")
]

# --- Output Schema + Parser ---
schemas = [
    ResponseSchema(name="answer", description="Final answer."),
    ResponseSchema(name="source", description="Tool(s) used.")
]
parser = StructuredOutputParser.from_response_schemas(schemas)
fmt = parser.get_format_instructions()

# --- LLM Setup ---
props = LoadProperties()
llm = ChatOCIGenAI(
    model_id="meta.llama-3.3-70b-instruct",
    service_endpoint=props.getEndpoint(),
    compartment_id=props.getCompartment(),
    auth_type="INSTANCE_PRINCIPAL",
    model_kwargs={"max_tokens": 600}
)

# --- Use Reliable Agent Type for OCI ---
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose=True,
    agent_kwargs={"system_message": f"""
You are a helpful agent. Use Calculator or Wikipedia to answer questions.

Always include the tool(s) used in your final JSON response, under the 'source' key.

Return ONLY in this JSON format:
{fmt}

Examples:
{{"answer": "17 * 24 is 408", "source": "Calculator"}}
{{"answer": "Hans Lippershey invented the telescope", "source": "Wikipedia"}}
"""}
)

# --- Run Agent + Display as Markdown Table ---
res = agent.invoke({"input": "What is 17 * 24 and who invented the telescope?"})
try:
    out = parser.parse(res["output"])
except:
    # Minimal fallback: fix with the LLM if needed
    repaired = llm.invoke(f"""
Format this into JSON:
{fmt}

Output:
{res['output']}
""")
    out = parser.parse(repaired.content if hasattr(repaired, "content") else str(repaired))

md = f"""| Field   | Value |
|---------|-------|
| Answer  | {out['answer']} |
| Source  | {out['source']} |"""
display(Markdown(md))
